In [41]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.tools import tool

In [42]:
@tool
def multiply(a: int, b: int) -> int:
    """
    Use to Multiply a and b
    Args: 
    a: first int
    b: second int
    """
    return int(a * b)

In [45]:
@tool
def add(a: int, b: int)-> int:
    """
    Use to Add a and b
    Args: 
    a: first int
    b: second int
    """
    return a + b

In [44]:
@tool
def subtract(a: int, b: int)-> int:
    """
    Use to Subtract a and b
    Args: 
    a: first int
    b: second int
    """
    return a - b

In [43]:
@tool
def divide(a: int, b: int)-> int:
    """
    Use to Divide a and b
    Args: 
    a: first int
    b: second int
    """
    return a // b

In [39]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langgraph.graph.message import MessagesState

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict


In [47]:
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

tools = [multiply, add, subtract, divide]

In [46]:
llm = ChatGroq(model = "qwen/qwen3-32b", max_retries = 2, temperature = 0.7)
llm_with_tools = llm.bind_tools(tools)

system_prompt = [
    SystemMessage(
        """
        Make use of the available tools.
        """
    )
]



In [ ]:
def tool_calling_llm(state: MessagesState):
    return {"messages" : [llm_with_tools.invoke(system_prompt + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage(content="What is the multiple of 2 and 22?")]})
print(result["messages"][-1].pretty_print())

================================== Ai Message ==================================

The product of 2 and 22 is $\boxed{44}$.
None


In [63]:
list_of_response = result.get("messages")
for i in list_of_response:
    print(i.pretty_print())

================================ Human Message =================================

What is the multiple of 2 and 22?
None
================================== Ai Message ==================================
Tool Calls:
  multiply (dh861pc02)
 Call ID: dh861pc02
  Args:
    a: 2
    b: 22
None
================================= Tool Message =================================
Name: multiply

44
None
================================== Ai Message ==================================

The product of 2 and 22 is $\boxed{44}$.
None
